# Evolution Simulation - Interactive Analysis

This notebook provides an interactive environment for analyzing your simulation data.

**Quick Start:**
1. Run all cells with "Run All"
2. Modify parameters and re-run specific analyses
3. Export custom visualizations

**Requirements:**
```bash
pip install pandas numpy matplotlib seaborn scipy scikit-learn jupyter
```

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Configure display
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## 1. Load Data

In [2]:
# Auto-detect most recent CSV files
data_dir = Path("..")

# Find files
entity_files = list(data_dir.glob("entity_details_*.csv"))
biome_files = list(data_dir.glob("biome_details_*.csv"))
batch_files = list(data_dir.glob("*batch_results*.csv"))

# Load most recent
if entity_files:
    entity_file = max(entity_files, key=lambda p: p.stat().st_mtime)
    entity_data = pd.read_csv(entity_file)
    print(f"✅ Loaded entity data: {entity_file.name}")
    print(f"   Records: {len(entity_data):,}")
else:
    entity_data = None
    print("⚠️ No entity details file found")

if biome_files:
    biome_file = max(biome_files, key=lambda p: p.stat().st_mtime)
    biome_data = pd.read_csv(biome_file)
    print(f"✅ Loaded biome data: {biome_file.name}")
    print(f"   Records: {len(biome_data):,}")
else:
    biome_data = None
    print("⚠️ No biome details file found")

if batch_files:
    batch_file = max(batch_files, key=lambda p: p.stat().st_mtime)
    batch_data = pd.read_csv(batch_file)
    print(f"✅ Loaded batch data: {batch_file.name}")
    print(f"   Simulations: {len(batch_data):,}")
else:
    batch_data = None
    print("⚠️ No batch results file found")

⚠️ No entity details file found
⚠️ No biome details file found
⚠️ No batch results file found


## 2. Quick Data Overview

In [3]:
# Display first few rows
if entity_data is not None:
    print("Entity Data Sample:")
    display(entity_data.head())
    print("\nColumns:", list(entity_data.columns))
    print("\nData types:")
    display(entity_data.dtypes)

In [ ]:
# Quick statistics
if entity_data is not None:
    print("Entity Data Statistics:")
    display(entity_data.describe())

## 3. Population Dynamics

In [ ]:
if entity_data is not None:
    # Population over time
    population_over_time = entity_data.groupby('Step')['EntityID'].nunique()
    
    plt.figure(figsize=(12, 5))
    plt.plot(population_over_time.index, population_over_time.values, linewidth=2)
    plt.xlabel('Simulation Step', fontsize=12)
    plt.ylabel('Population Count', fontsize=12)
    plt.title('Population Dynamics Over Time', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Peak population: {population_over_time.max()}")
    print(f"Final population: {population_over_time.iloc[-1]}")
    print(f"Steps simulated: {population_over_time.index.max()}")

## 4. Trait Distribution Analysis

In [ ]:
if entity_data is not None:
    # Trait distributions
    traits = ['Speed', 'Mass', 'EnergyEfficiency', 'SightRange', 'MetabolismRate', 'ReproductionThreshold']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Trait Distribution Analysis', fontsize=16, fontweight='bold')
    
    for idx, trait in enumerate(traits):
        ax = axes[idx // 3, idx % 3]
        ax.hist(entity_data[trait], bins=30, edgecolor='black', alpha=0.7)
        ax.axvline(entity_data[trait].mean(), color='r', linestyle='--', linewidth=2, label='Mean')
        ax.axvline(entity_data[trait].median(), color='g', linestyle='--', linewidth=2, label='Median')
        ax.set_xlabel(trait, fontsize=11)
        ax.set_ylabel('Frequency', fontsize=11)
        ax.set_title(f'{trait} Distribution', fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Biome Analysis

In [ ]:
if entity_data is not None:
    # Biome distribution
    biome_counts = entity_data['BiomeType'].value_counts()
    
    plt.figure(figsize=(10, 6))
    biome_counts.plot(kind='barh', color=plt.cm.tab10(range(len(biome_counts))))
    plt.xlabel('Number of Observations', fontsize=12)
    plt.ylabel('Biome Type', fontsize=12)
    plt.title('Entity Observations by Biome Type', fontsize=14, fontweight='bold')
    plt.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\nBiome Distribution:")
    for biome, count in biome_counts.items():
        pct = (count / len(entity_data)) * 100
        print(f"{biome:15s}: {count:6,} ({pct:5.2f}%)")

## 6. Survival Analysis

In [ ]:
if entity_data is not None:
    # Calculate entity survival metrics
    entity_stats = entity_data.groupby('EntityID').agg({
        'Age': 'max',
        'Step': 'count',
        'Speed': 'mean',
        'Mass': 'mean',
        'EnergyEfficiency': 'mean',
        'SightRange': 'mean'
    }).reset_index()
    
    entity_stats.rename(columns={'Step': 'StepsSurvived'}, inplace=True)
    
    # Correlation with survival
    print("Correlation with Steps Survived:")
    print(entity_stats[['Speed', 'Mass', 'EnergyEfficiency', 'SightRange', 'StepsSurvived']].corr()['StepsSurvived'].sort_values(ascending=False))
    
    # Visualize top correlations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Speed vs Survival
    sns.regplot(data=entity_stats, x='Speed', y='StepsSurvived', ax=axes[0])
    axes[0].set_title('Speed vs Survival', fontweight='bold')
    axes[0].set_xlabel('Average Speed', fontsize=11)
    axes[0].set_ylabel('Steps Survived', fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # EnergyEfficiency vs Survival
    sns.regplot(data=entity_stats, x='EnergyEfficiency', y='StepsSurvived', ax=axes[1])
    axes[1].set_title('Energy Efficiency vs Survival', fontweight='bold')
    axes[1].set_xlabel('Average Energy Efficiency', fontsize=11)
    axes[1].set_ylabel('Steps Survived', fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Spatial Distribution

In [ ]:
if entity_data is not None:
    # Get last step for spatial snapshot
    last_step = entity_data['Step'].max()
    snapshot = entity_data[entity_data['Step'] == last_step]
    
    if len(snapshot) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(f'Spatial Distribution (Step {last_step})', fontsize=16, fontweight='bold')
        
        # Energy distribution
        scatter1 = axes[0].scatter(snapshot['X'], snapshot['Y'], c=snapshot['Energy'], 
                                  cmap='plasma', s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
        axes[0].set_xlabel('X Position', fontsize=11)
        axes[0].set_ylabel('Y Position', fontsize=11)
        axes[0].set_title('Energy Distribution', fontweight='bold')
        axes[0].set_aspect('equal')
        plt.colorbar(scatter1, ax=axes[0], label='Energy')
        
        # Age distribution
        scatter2 = axes[1].scatter(snapshot['X'], snapshot['Y'], c=snapshot['Age'], 
                                  cmap='viridis', s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
        axes[1].set_xlabel('X Position', fontsize=11)
        axes[1].set_ylabel('Y Position', fontsize=11)
        axes[1].set_title('Age Distribution', fontweight='bold')
        axes[1].set_aspect('equal')
        plt.colorbar(scatter2, ax=axes[1], label='Age')
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ No entities in final step for spatial analysis")

## 8. Custom Analysis

Add your own analysis below:

In [ ]:
# Your custom analysis here

## Summary

This notebook provides an interactive environment for exploring your simulation data. You can:

- Modify visualizations by changing parameters
- Add custom analyses in the last cell
- Export figures with `plt.savefig('filename.png', dpi=300, bbox_inches='tight')`
- Filter data with pandas queries
- Perform statistical tests with scipy

**For automated batch analysis, use the Python scripts instead:**
- `analyze_evolution.py` - Main analysis
- `batch_comparison.py` - Multi-seed comparison
- `statistical_analysis.py` - Advanced statistics